In [1]:
import pandas as pd
import numpy as np
import os

data_path = os.path.expanduser('~/Desktop/football-intelligence/data/')

# Chargement
players = pd.read_csv(data_path + 'players.csv')
valuations = pd.read_csv(data_path + 'player_valuations.csv')
transfers = pd.read_csv(data_path + 'transfers.csv')
appearances = pd.read_csv(data_path + 'appearances.csv')
clubs = pd.read_csv(data_path + 'clubs.csv')
competitions = pd.read_csv(data_path + 'competitions.csv')

# Afficher les ligues disponibles
print("=== LIGUES DISPONIBLES ===")
print(competitions[['competition_id','name','country_name']].head(30).to_string())

=== LIGUES DISPONIBLES ===
   competition_id                               name country_name
0              A1                         bundesliga      Austria
1            AFAC                      afc-asian-cup          NaN
2            AFCN              africa-cup-of-nations          NaN
3            ARG1                    torneo-apertura    Argentina
4            AUS1                       a-league-men    Australia
5             BE1                 jupiler-pro-league      Belgium
6            BESC                volkswagen-supercup      Belgium
7            BRA1      campeonato-brasileiro-serie-a       Brazil
8              C1                       super-league  Switzerland
9             CDR                       copa-del-rey        Spain
10            CGB                            efl-cup      England
11            CIT                          italy-cup        Italy
12             CL              uefa-champions-league          NaN
13            CLQ   uefa-champions-league-qualify

In [2]:
# Recherche directe des Top 5 ligues
top5_keywords = ['premier-league', 'laliga', 'bundesliga', 'serie-a', 'ligue-1']
top5 = competitions[competitions['name'].isin(top5_keywords)]
print(top5[['competition_id', 'name', 'country_name']].to_string())

   competition_id            name country_name
0              A1      bundesliga      Austria
23            ES1          laliga        Spain
27            FR1         ligue-1       France
29            GB1  premier-league      England
33            IT1         serie-a        Italy
37             L1      bundesliga      Germany


In [3]:
# IDs officiels des Top 5 ligues
TOP5_IDS = ['GB1', 'ES1', 'FR1', 'IT1', 'L1']

# Filtrer les clubs des Top 5 ligues
clubs_top5 = clubs[clubs['domestic_competition_id'].isin(TOP5_IDS)].copy()
print(f"Clubs Top 5 ligues : {len(clubs_top5)}")

# Filtrer les joueurs avec valeur marchande connue
players_clean = players[players['market_value_in_eur'].notna()].copy()
print(f"Joueurs avec valeur marchande : {len(players_clean)}")

# Filtrer les apparences des Top 5 ligues uniquement
appearances_top5 = appearances[appearances['competition_id'].isin(TOP5_IDS)].copy()
print(f"Apparences Top 5 ligues : {len(appearances_top5):,}")

# Filtrer les transferts avec montant connu
transfers_clean = transfers[transfers['transfer_fee'].notna()].copy()
print(f"Transferts avec montant : {len(transfers_clean):,}")

print("\n✅ Données filtrées avec succès !")
print(f"\nClubs disponibles par ligue :")
print(clubs_top5.groupby('domestic_competition_id').size().rename('nb_clubs'))

Clubs Top 5 ligues : 176
Joueurs avec valeur marchande : 33025
Apparences Top 5 ligues : 708,901
Transferts avec montant : 27,517

✅ Données filtrées avec succès !

Clubs disponibles par ligue :
domestic_competition_id
ES1    33
FR1    36
GB1    37
IT1    39
L1     31
Name: nb_clubs, dtype: int64


In [5]:
# Vérification des colonnes disponibles
print("=== COLONNES PLAYERS ===")
print(list(players_clean.columns))

print("\n=== COLONNES CLUBS ===")
print(list(clubs_top5.columns))

print("\n=== COLONNES APPEARANCES ===")
print(list(appearances_top5.columns))

=== COLONNES PLAYERS ===
['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']

=== COLONNES CLUBS ===
['club_id', 'club_code', 'name', 'domestic_competition_id', 'total_market_value', 'squad_size', 'average_age', 'foreigners_number', 'foreigners_percentage', 'national_team_players', 'stadium_name', 'stadium_seats', 'net_transfer_record', 'coach_name', 'last_season', 'filename', 'url']

=== COLONNES APPEARANCES ===
['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'date', 'player_name', 'competition_id', 'yellow_cards', 'red_cards',

In [7]:
# Construction du profil joueur - version finale corrigée
player_stats = appearances_top5.groupby('player_id').agg(
    total_appearances=('appearance_id', 'count'),
    total_goals=('goals', 'sum'),
    total_assists=('assists', 'sum'),
    total_minutes=('minutes_played', 'sum'),
    total_yellow=('yellow_cards', 'sum'),
    total_red=('red_cards', 'sum')
).reset_index()

# Stats par 90 minutes
player_stats['goals_per90'] = (player_stats['total_goals'] / player_stats['total_minutes'] * 90).round(2)
player_stats['assists_per90'] = (player_stats['total_assists'] / player_stats['total_minutes'] * 90).round(2)
player_stats['minutes_per_game'] = (player_stats['total_minutes'] / player_stats['total_appearances']).round(1)

# Fusion avec infos joueurs
player_profile = player_stats.merge(
    players_clean[['player_id', 'name', 'position', 'sub_position',
                   'market_value_in_eur', 'highest_market_value_in_eur',
                   'current_club_id', 'date_of_birth', 'foot',
                   'height_in_cm', 'country_of_citizenship',
                   'current_club_domestic_competition_id']],
    on='player_id', how='inner'
)

# Calcul de l'âge - on ignore les dates nulles
player_profile['date_of_birth'] = pd.to_datetime(player_profile['date_of_birth'], errors='coerce')
player_profile['age'] = player_profile['date_of_birth'].apply(
    lambda x: int((pd.Timestamp.now() - x).days / 365.25) if pd.notna(x) else np.nan
)

# Fusion avec club actuel
player_profile = player_profile.merge(
    clubs_top5[['club_id', 'name', 'domestic_competition_id',
                'squad_size', 'average_age', 'total_market_value', 'net_transfer_record']],
    left_on='current_club_id', right_on='club_id', how='left',
    suffixes=('_player', '_club')
)

print(f"✅ Profils joueurs construits : {len(player_profile):,}")
print(f"\nAperçu :")
print(player_profile[['name_player', 'position', 'age', 'goals_per90',
                       'assists_per90', 'market_value_in_eur',
                       'name_club', 'domestic_competition_id']].head(10).to_string())

✅ Profils joueurs construits : 10,101

Aperçu :
           name_player    position   age  goals_per90  assists_per90  market_value_in_eur                                  name_club domestic_competition_id
0       Miroslav Klose      Attack  47.0         0.51           0.28            1000000.0              Società Sportiva Lazio S.p.A.                     IT1
1   Roman Weidenfeller  Goalkeeper  45.0         0.00           0.00             750000.0                          Borussia Dortmund                      L1
2     Dimitar Berbatov      Attack  45.0         0.42           0.16            1000000.0                                        NaN                     NaN
3                Lúcio    Defender  47.0         0.00           0.00             200000.0                     Juventus Football Club                     IT1
4           Tom Starke  Goalkeeper  45.0         0.00           0.00             100000.0                          FC Bayern München                      L1
5  Christo

In [8]:
# Sauvegarde du profil joueur propre
save_path = os.path.expanduser('~/Desktop/football-intelligence/data/')

player_profile.to_csv(save_path + 'player_profile_clean.csv', index=False)
clubs_top5.to_csv(save_path + 'clubs_top5_clean.csv', index=False)

print("✅ Fichiers sauvegardés !")
print(f"   - player_profile_clean.csv ({len(player_profile):,} joueurs)")
print(f"   - clubs_top5_clean.csv ({len(clubs_top5)} clubs)")

# Résumé final du nettoyage
print("\n=== RÉSUMÉ NETTOYAGE ===")
print(f"Joueurs avec position connue : {player_profile['position'].notna().sum():,}")
print(f"Joueurs avec valeur marchande : {player_profile['market_value_in_eur'].notna().sum():,}")
print(f"Joueurs avec club identifié : {player_profile['name_club'].notna().sum():,}")
print(f"Âge moyen des joueurs : {player_profile['age'].mean():.1f} ans")
print(f"Valeur marchande moyenne : {player_profile['market_value_in_eur'].mean():,.0f} €")

✅ Fichiers sauvegardés !
   - player_profile_clean.csv (10,101 joueurs)
   - clubs_top5_clean.csv (176 clubs)

=== RÉSUMÉ NETTOYAGE ===
Joueurs avec position connue : 10,101
Joueurs avec valeur marchande : 10,101
Joueurs avec club identifié : 7,163
Âge moyen des joueurs : 31.9 ans
Valeur marchande moyenne : 4,145,090 €
